In [ ]:
"""
Per-pixel spectral response, the baseline 04.00 offset, and matching against a
spectral library.

WHAT THIS IS FOR. Three questions, in the order they have to be answered:

  1. Is the Sentinel-2 BOA_ADD_OFFSET applied? Since processing baseline 04.00
     (25 January 2022) L2A products carry a band-dependent offset of -1000, so
     reflectance is (DN + offset) / 10000. sidecar/infer.py divides by 10000
     with no offset term. Nothing downstream is trustworthy until this is
     settled, because every normalised index compresses toward zero when a
     constant is added to both of its bands.

  2. What does the spectrum of each predicted class look like, and how wide is
     it within one AOI? This is the reading the application does not currently
     expose, and it is what would explain a domain-shift number band by band
     rather than as a single distance.

  3. Can a pixel be matched against a spectral library? Only after the library
     spectrum is convolved with the Sentinel-2 spectral response functions,
     since a library is hyperspectral and this sensor has a handful of broad
     bands.

HOW IT RUNS. This imports the sidecar's own loaders rather than reimplementing
them, so what is measured here is the path the application actually takes. Run
it as a script, or convert it with jupytext if you want cells:

    .venv/bin/python experiments/spectral_response_and_offset.py

Figures are written next to this file. Nothing here writes into the app.
"""

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path(__file__).resolve().parent.parent
# The sidecar's modules import each other flat (`from class_palette import ...`),
# so it is that directory that goes on the path, not the repository root.
sys.path.insert(0, str(REPO / "sidecar"))
OUT = Path(__file__).resolve().parent / "figures"
OUT.mkdir(exist_ok=True)

from infer import (  # noqa: E402
    list_stac_products,
    load_band_to_reference_grid,
    load_and_clip_band,
)

import matplotlib.pyplot as plt  # noqa: E402
import scienceplots  # noqa: F401,E402

plt.style.use(["science", "no-latex"])

# The offset the baseline introduced. Reflectance is (DN + BOA_ADD_OFFSET) / QV.
BOA_ADD_OFFSET = -1000.0
QUANTIFICATION_VALUE = 10000.0

# Band centres in nm, for plotting a spectrum in wavelength order rather than
# in the arbitrary order the band names sort in.
BAND_NM = {
    "B02": 492.4, "B03": 559.8, "B04": 664.6, "B05": 704.1, "B06": 740.5,
    "B07": 782.8, "B08": 832.8, "B8A": 864.7, "B11": 1613.7, "B12": 2202.4,
}
# What the sidecar reads today: four at 10 m, three at 20 m.
TERRA_BANDS = [("B02", "10m"), ("B03", "10m"), ("B04", "10m"), ("B08", "10m"),
               ("B8A", "20m"), ("B11", "20m"), ("B12", "20m")]

# A study area in Cascavel, Parana, which is where the shipped models were
# fitted. Any polygon works; this one keeps the test on ground the classifier
# is entitled to speak about.
from shapely.geometry import shape  # noqa: E402

AOI_GEOJSON = {
    "type": "Polygon",
    "coordinates": [[
        [-53.52, -24.92], [-53.46, -24.92], [-53.46, -24.87],
        [-53.52, -24.87], [-53.52, -24.92],
    ]],
}
# The sidecar's loaders take a shapely geometry, not the GeoJSON the API speaks.
AOI = shape(AOI_GEOJSON)
START, END = "2025-08-16", "2026-08-16"
MAX_CLOUD = 40.0

In [ ]:
# ---------------------------------------------------------------- discovery
def discover(start=START, end=END, max_cloud=MAX_CLOUD, monthly=True):
    """The same scene search the application runs, with the same defaults."""
    products = list_stac_products(AOI, start, end, max_cloud=max_cloud,
                                  monthly_best=monthly)
    rows = [{
        "id": p.get("id", "?"),
        "date": str(p.get("datetime", ""))[:10],
        "cloud": round(float(p.get("cloud_cover", float("nan"))), 1),
        "baseline": p.get("properties", {}).get("s2:processing_baseline", "?"),
    } for p in products]
    return products, pd.DataFrame(rows)


products, scenes = discover()
print(f"{len(products)} scenes over the AOI")
print(scenes.to_string(index=False))

In [ ]:
# ------------------------------------------------- 1. is the offset applied?
def read_stack(product, bands=TERRA_BANDS):
    """
    Raw DN for one scene on a common grid, as the sidecar reads it.

    Returned as DN rather than reflectance on purpose: the whole question is
    what the conversion should be, so the conversion is not done here.
    """
    ref, ref_prof = load_and_clip_band(product, "B04", AOI, "10m")
    out = {}
    for name, res in bands:
        out[name] = load_band_to_reference_grid(product, name, AOI, ref_prof,
                                                resolution=res).astype("float64")
    return out, ref_prof


def to_reflectance(dn, apply_offset):
    """Both conventions, so they can be compared rather than argued about."""
    v = (dn + BOA_ADD_OFFSET) / QUANTIFICATION_VALUE if apply_offset \
        else dn / QUANTIFICATION_VALUE
    return v


scene = products[len(products) // 2]
stack, prof = read_stack(scene)
baseline = scene.get("properties", {}).get("s2:processing_baseline", "?")
print(f"scene {scene.get('id')}  baseline {baseline}")

valid = np.ones_like(stack["B04"], dtype=bool)
for a in stack.values():
    valid &= np.isfinite(a) & (a > 0)
print(f"{valid.sum()} valid pixels of {valid.size}")

summary = []
for name, dn in stack.items():
    v = dn[valid]
    summary.append({
        "band": name, "nm": BAND_NM[name], "DN_median": np.median(v),
        "rho_no_offset": np.median(to_reflectance(v, False)),
        "rho_with_offset": np.median(to_reflectance(v, True)),
    })
spec = pd.DataFrame(summary).sort_values("nm")
print(spec.to_string(index=False, float_format=lambda x: f"{x:8.3f}"))

# A reflectance above roughly 0.15 in the red over a vegetated field is not
# physically plausible; over dense canopy the red sits near 0.03 because
# chlorophyll absorbs there. That is the check this figure is for.
fig, ax = plt.subplots(figsize=(3.5, 2.6))
ax.plot(spec["nm"], spec["rho_no_offset"], "o-", lw=1.0, ms=3,
        label="DN / 10000")
ax.plot(spec["nm"], spec["rho_with_offset"], "s--", lw=1.0, ms=3,
        label="(DN - 1000) / 10000")
ax.set_xlabel(r"Wavelength (nm)")
ax.set_ylabel(r"Surface reflectance (dimensionless)")
ax.set_title(f"AOI median spectrum, baseline {baseline}")
ax.legend(frameon=False, fontsize=7)
fig.savefig(OUT / "offset_spectrum.pdf", dpi=300, bbox_inches="tight")
fig.savefig(OUT / "offset_spectrum.png", dpi=300, bbox_inches="tight")
plt.close(fig)

In [ ]:
# ------------------------------------------- the effect on a normalised index
def ndvi(red, nir):
    d = nir + red
    return np.where(d > 0, (nir - red) / d, np.nan)


red_no, nir_no = (to_reflectance(stack["B04"], False),
                  to_reflectance(stack["B08"], False))
red_yes, nir_yes = (to_reflectance(stack["B04"], True),
                    to_reflectance(stack["B08"], True))
n_no, n_yes = ndvi(red_no, nir_no)[valid], ndvi(red_yes, nir_yes)[valid]

print(f"NDVI without offset : median {np.median(n_no):.3f}  p95 {np.percentile(n_no, 95):.3f}")
print(f"NDVI with offset    : median {np.median(n_yes):.3f}  p95 {np.percentile(n_yes, 95):.3f}")
print(f"median shift        : {np.median(n_yes - n_no):+.3f}")

# Adding a constant c to both bands leaves the numerator alone and inflates the
# denominator by 2c, so the compression is worst where NDVI is highest. Plotting
# one against the other shows that directly.
fig, ax = plt.subplots(figsize=(3.5, 2.6))
idx = np.random.default_rng(0).choice(n_no.size, size=min(20000, n_no.size),
                                      replace=False)
ax.scatter(n_no[idx], n_yes[idx], s=0.6, alpha=0.25, edgecolors="none")
lim = [min(n_no.min(), n_yes.min()), 1.0]
ax.plot(lim, lim, "k-", lw=0.6)
ax.set_xlabel(r"NDVI from DN / 10000")
ax.set_ylabel(r"NDVI with the offset applied")
ax.set_title("Compression toward zero, worst at high NDVI")
fig.savefig(OUT / "offset_ndvi.pdf", dpi=300, bbox_inches="tight")
fig.savefig(OUT / "offset_ndvi.png", dpi=300, bbox_inches="tight")
plt.close(fig)

In [ ]:
# ------------------------------ 2. spectrum per class, with its own dispersion
# Classes come from the shipped model, so what is described is what the
# application would report, not an independent segmentation.
import joblib  # noqa: E402

MODEL = REPO / "model"
rf = joblib.load(MODEL / "rf_classifier.joblib")
scaler = joblib.load(MODEL / "scaler.joblib")
le = joblib.load(MODEL / "label_encoder.joblib")
print("classes:", list(le.classes_), "| features:", rf.n_features_in_)


def class_map(apply_offset):
    """
    Run the shipped classifier over the AOI under one reflectance convention.

    build_feature_matrix hardcodes DN / 10000, so it cannot be reused to answer
    a question about that constant. The features are rebuilt here from the same
    definitions, which is the one place this file duplicates the sidecar and the
    reason it does.
    """
    from infer import build_feature_matrix
    # 22 raw NDVI dates, padded or truncated to that length whatever the scene
    # count. infer.py derives it as len(feature_names) - 58, which is the same
    # number and the safer source, since it follows the shipped artifact.
    feature_names = joblib.load(MODEL / "feature_names.joblib")
    n_dates_model = len(feature_names) - 58
    if not apply_offset:
        return build_feature_matrix(products, AOI, prof, n_dates_model)
    raise NotImplementedError(
        "Rebuilding the 80-feature matrix under the corrected convention is the "
        "next step; it needs the temporal descriptors, not just the bands. Run "
        "the uncorrected path first and read the per-class spectra below, which "
        "do not depend on it."
    )


X, mask = class_map(apply_offset=False)
if X is None:
    raise SystemExit("no feature matrix; widen the period or raise MAX_CLOUD")
pred = le.inverse_transform(rf.predict(scaler.transform(X)))
conf = rf.predict_proba(scaler.transform(X)).max(axis=1)
print(f"{len(pred)} classified pixels; mean confidence {conf.mean():.3f}")

# Per-class spectrum on the single scene read above, under both conventions.
rows = []
flat_valid = mask.reshape(-1)
for band in [b for b, _ in TERRA_BANDS]:
    dn_flat = stack[band].reshape(-1)[flat_valid]
    for convention, on in (("DN/10000", False), ("offset applied", True)):
        rho = to_reflectance(dn_flat, on)
        for cls in np.unique(pred):
            sel = (pred == cls) & np.isfinite(rho)
            if sel.sum() < 30:
                continue
            rows.append({
                "class": str(cls), "band": band, "nm": BAND_NM[band],
                "convention": convention, "n": int(sel.sum()),
                "mean": float(np.mean(rho[sel])),
                "sd": float(np.std(rho[sel])),
                "p05": float(np.percentile(rho[sel], 5)),
                "p95": float(np.percentile(rho[sel], 95)),
            })

sig = pd.DataFrame(rows)
sig.to_csv(OUT.parent / "class_spectra.csv", index=False)
print(sig[sig.convention == "offset applied"]
      .pivot_table(index="class", columns="band", values="mean")
      .to_string(float_format=lambda x: f"{x:6.3f}"))

fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.7), sharey=True)
for ax, convention in zip(axes, ["DN/10000", "offset applied"]):
    part = sig[sig.convention == convention]
    for cls, g in part.groupby("class"):
        g = g.sort_values("nm")
        ax.plot(g["nm"], g["mean"], "-o", lw=0.9, ms=2.5, label=cls)
        ax.fill_between(g["nm"], g["p05"], g["p95"], alpha=0.12, linewidth=0)
    ax.set_xlabel(r"Wavelength (nm)")
    ax.set_title(convention)
axes[0].set_ylabel(r"Surface reflectance (dimensionless)")
axes[1].legend(frameon=False, fontsize=6)
fig.savefig(OUT / "class_spectra.pdf", dpi=300, bbox_inches="tight")
fig.savefig(OUT / "class_spectra.png", dpi=300, bbox_inches="tight")
plt.close(fig)

In [ ]:
# --------------------------------- 3. matching against a spectral library
# A library spectrum is hyperspectral and this sensor is not, so the two are
# not comparable until the library is passed through the sensor's own response.
# ESA publishes those functions; the file is fetched rather than transcribed,
# because transcribing 13 response curves by hand is how a silent error enters.
SRF_URLS = [
    "https://sentinels.copernicus.eu/documents/247904/685211/"
    "S2-SRF_COPE-GSEG-EOPG-TN-15-0007_3.1.xlsx",
    "https://landsat.usgs.gov/landsat/spectral_viewer/bands/"
    "Sentinel-2A%20MSI%20Spectral%20Responses.xlsx",
]
SRF_PATH = OUT.parent / "s2_srf.xlsx"


def fetch_srf():
    """
    Download the spectral response functions, or say why it could not.

    Two sources are tried because ESA reorganises its document library and the
    direct link rots; the USGS spectral viewer mirrors the same measurements.
    Nothing is substituted on failure: without the real curves there is no
    honest convolution, so the section declines to run.
    """
    if SRF_PATH.exists():
        return SRF_PATH
    import urllib.request
    for url in SRF_URLS:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as r, \
                    open(SRF_PATH, "wb") as f:
                f.write(r.read())
            print(f"SRF from {url.split('/')[2]}")
            return SRF_PATH
        except Exception as exc:  # noqa: BLE001
            print(f"  {url.split('/')[2]}: {type(exc).__name__}: {exc}")
    print("No SRF available. Fetch it by hand from the ESA document library")
    print(f"and save it as {SRF_PATH}")
    return None


def convolve(wavelength_nm, reflectance, srf_df, band, satellite="S2A"):
    """
    What this sensor would report looking at a hyperspectral spectrum.

        rho_band = integral(rho(l) * S(l) dl) / integral(S(l) dl)

    Anything outside the library's own wavelength range is dropped rather than
    extrapolated, and the band is refused when the library does not cover it,
    since a partially covered band is a number with no defined meaning.
    """
    # The sheet names bands without the leading zero: B02 is B2, while B8A and
    # B11 keep their own form.
    short = f"B{int(band[1:])}" if band[1:].isdigit() else band
    col = f"{satellite}_SR_AV_{short}"
    if col not in srf_df.columns:
        raise KeyError(f"{col} not in the SRF sheet: {list(srf_df.columns)[:6]}")
    s_wl = srf_df["SR_WL"].to_numpy(dtype=float)
    s = srf_df[col].to_numpy(dtype=float)
    keep = s > 1e-6
    s_wl, s = s_wl[keep], s[keep]
    if s_wl.min() < np.min(wavelength_nm) or s_wl.max() > np.max(wavelength_nm):
        raise ValueError(f"library does not cover {band} ({s_wl.min():.0f}"
                         f"-{s_wl.max():.0f} nm)")
    r = np.interp(s_wl, wavelength_nm, reflectance)
    return float(np.trapezoid(r * s, s_wl) / np.trapezoid(s, s_wl))


def spectral_angle(a, b):
    """
    Spectral Angle Mapper, in radians. Scale-invariant, which is why it is the
    standard choice here: a pixel in shadow differs from the same material in
    sun by a multiplier, and the angle ignores exactly that.
    """
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(np.arccos(np.clip(
        np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)), -1, 1)))


srf_file = fetch_srf()
if srf_file is None:
    print("\nSkipping the library section: no SRF available.")
else:
    srf = pd.read_excel(srf_file, sheet_name="Spectral Responses (S2A)")
    print(f"SRF: {len(srf)} rows, {srf['SR_WL'].min():.0f}-{srf['SR_WL'].max():.0f} nm")

    # Prove the convolution rather than assert it. A spectrum that is flat at
    # r must come back as r in every band, whatever the response shape, since
    # the integral is normalised by the response's own area. Any weighting or
    # units error breaks this.
    wl = np.arange(380.0, 2500.0, 1.0)
    for r in (0.25, 0.60):
        got = {b: convolve(wl, np.full_like(wl, r), srf, b)
               for b, _ in TERRA_BANDS}
        worst = max(abs(v - r) for v in got.values())
        print(f"  flat spectrum at {r}: max deviation {worst:.2e}")
        assert worst < 1e-9, got

    # The bandwidth each reading actually integrates over, which is the reason
    # a library cannot be compared to these numbers channel by channel.
    for band, _ in TERRA_BANDS:
        short = f"B{int(band[1:])}" if band[1:].isdigit() else band
        s = srf[f"S2A_SR_AV_{short}"].to_numpy(float)
        w = srf["SR_WL"].to_numpy(float)
        keep = s > 0.01 * s.max()
        print(f"  {band}: {w[keep].min():7.1f} to {w[keep].max():7.1f} nm"
              f"  ({w[keep].max() - w[keep].min():5.1f} nm wide)")

    print("\nTo match a library: download ECOSTRESS (speclib.jpl.nasa.gov) or")
    print("EcoSIS (ecosis.org), call convolve() once per band to get its")
    print("7-vector, then spectral_angle() against the per-class means in")
    print("class_spectra.csv. With seven broad bands the angle separates")
    print("vegetation, soil, water and built surfaces; it does not identify a")
    print("species. Read a small angle as consistency, not identification.")

print(f"\nFigures and tables in {OUT.parent}")